# Chapter 35 Guided To-Do Practice Labs

**Book Reference:** Chapter 35 (Part VII — Research Capstone), Section 35.5 (Guided To-Do Practice)  
Run **Step 0** to activate `sharukh-tutor`, implement all 6 reproducible-research and deployment components (payload validation contract, `ClinicalRiskService`, Model Card generator, SHA-256 bundle builder, bundle integrity verifier, and TRIPOD+AI compliance scorer), and pass each **function or class name (without parentheses!)** to `tutor.check(...)`. Use `tutor.hint("ch35_todo_XX", level=1..3)` whenever you want a progressive nudge.

In [ ]:
# @title Step 0: Click Play (▶) to Activate Interactive Hints & Auto-Checker
!pip install -q sharukh-tutor
from datetime import datetime, timezone
import hashlib
import json
import math
from pathlib import Path
import platform
import numpy as np
import pandas as pd
import sharukh_tutor as tutor

tutor.activate(chapter="ch35", notebook="guided_todos")


---
## To-Do 35.1 (Clinical Payload Contract): `validate_clinical_payload(payload)`

Implement `validate_clinical_payload(payload)` to enforce a strict clinical input contract:
- `payload` must be a `dict` containing all 5 keys: `{"maternal_age_years", "anc_visits", "maternal_hb_g_dl", "residence", "wealth_quintile"}`.
- Numeric fields must be finite `int`/`float` (NEVER `bool`!) inside inclusive physiological bounds: `maternal_age_years` in `[12, 60]`, `anc_visits` in `[0, 25]`, `maternal_hb_g_dl` in `[2.0, 25.0]`.
- String fields must be `str`, stripped & title-cased: `residence` in `{"Rural", "Urban"}`, `wealth_quintile` in `{"Poorest", "Poorer", "Middle", "Richer", "Richest"}`.
- Raise `ValueError` on any violation; otherwise return a clean normalized **copy** without mutating `payload`.

In [ ]:
def validate_clinical_payload(payload):
    """Validate & normalize a maternal risk request without mutation."""
    # STUB: raise NotImplementedError("Replace with your solution")
    ### BEGIN SOLUTION
    if not isinstance(payload, dict):
        raise ValueError("Payload must be a dictionary.")
    required = {
        "maternal_age_years",
        "anc_visits",
        "maternal_hb_g_dl",
        "residence",
        "wealth_quintile",
    }
    missing = sorted(required - set(payload.keys()))
    if missing:
        raise ValueError(f"Missing required keys: {missing}")

    bounds = [
        ("maternal_age_years", 12, 60),
        ("anc_visits", 0, 25),
        ("maternal_hb_g_dl", 2.0, 25.0),
    ]
    clean = {}
    for key, lo, hi in bounds:
        val = payload[key]
        if (
            isinstance(val, bool)
            or not isinstance(val, (int, float))
            or not math.isfinite(float(val))
        ):
            raise ValueError(f"{key} must be a finite number.")
        num = float(val)
        if not (lo <= num <= hi):
            raise ValueError(f"{key} out of range [{lo}, {hi}]: {num}")
        clean[key] = num

    res = payload["residence"]
    if not isinstance(res, str):
        raise ValueError("residence must be a string.")
    res_norm = res.strip().title()
    if res_norm not in {"Rural", "Urban"}:
        raise ValueError(f"Invalid residence: {res!r}")
    clean["residence"] = res_norm

    wq = payload["wealth_quintile"]
    if not isinstance(wq, str):
        raise ValueError("wealth_quintile must be a string.")
    wq_norm = wq.strip().title()
    valid_wq = {"Poorest", "Poorer", "Middle", "Richer", "Richest"}
    if wq_norm not in valid_wq:
        raise ValueError(f"Invalid wealth_quintile: {wq!r}")
    clean["wealth_quintile"] = wq_norm
    return clean
    ### END SOLUTION


In [ ]:
tutor.check("ch35_todo_01", validate_clinical_payload)


---
## To-Do 35.2 (Inference Service Wrapper): `ClinicalRiskService`

Build a versioned inference class `ClinicalRiskService`:
- `__init__(self, pipeline, threshold=0.35, model_version="v1.0")`: store `self.pipeline`, `self.threshold = float(threshold)`, and `self.model_version = str(model_version)`.
- `predict_one(self, payload)`: validate via `validate_clinical_payload(payload)`, build a 1-row `pd.DataFrame`, compute `prob = float(self.pipeline.predict_proba(row)[0, 1])`, assign `predicted_label = int(prob >= self.threshold)` and `risk_tier` (`"High"` if `prob >= threshold`, else `"Borderline"` if `prob >= threshold - 0.15`, else `"Low"`), and return `{"probability": round4, "predicted_label": int, "risk_tier": str, "threshold": float, "model_version": str}`.

In [ ]:
class ClinicalRiskService:
    """Versioned clinical risk service with input contract & tiers."""

    def __init__(self, pipeline, threshold=0.35, model_version="v1.0"):
        # STUB: raise NotImplementedError("Replace with your solution")
        ### BEGIN SOLUTION
        self.pipeline = pipeline
        self.threshold = float(threshold)
        self.model_version = str(model_version)
        ### END SOLUTION

    def predict_one(self, payload):
        # STUB: raise NotImplementedError("Replace with your solution")
        ### BEGIN SOLUTION
        clean = validate_clinical_payload(payload)
        row = pd.DataFrame([clean])
        prob = float(self.pipeline.predict_proba(row)[0, 1])
        label = int(prob >= self.threshold)
        if prob >= self.threshold:
            tier = "High"
        elif prob >= self.threshold - 0.15:
            tier = "Borderline"
        else:
            tier = "Low"
        return {
            "probability": round(prob, 4),
            "predicted_label": label,
            "risk_tier": tier,
            "threshold": self.threshold,
            "model_version": self.model_version,
        }
        ### END SOLUTION


In [ ]:
tutor.check("ch35_todo_02", ClinicalRiskService)


---
## To-Do 35.3 (Programmatic Model Card): `generate_model_card_markdown(meta)`

Implement `generate_model_card_markdown(meta)` returning a structured Markdown string containing these exact 5 level-2 headings:
- `## Model Details`
- `## Intended Use`
- `## Training Data`
- `## Evaluation Metrics`
- `## Fairness & Limitations`

Populate them from `meta` keys `name`, `version`, `intended_use`, `dataset`, `roc_auc`, `brier_score`, and `limitations`.

In [ ]:
def generate_model_card_markdown(meta):
    """Render a structured 5-section clinical Model Card string."""
    # STUB: raise NotImplementedError("Replace with your solution")
    ### BEGIN SOLUTION
    lims = meta["limitations"]
    if isinstance(lims, list):
        lim_text = "\n".join(f"- {item}" for item in lims)
    else:
        lim_text = str(lims)
    return (
        f"# Model Card: {meta['name']}\n\n"
        f"## Model Details\n"
        f"- **Name:** {meta['name']}\n"
        f"- **Version:** {meta['version']}\n\n"
        f"## Intended Use\n"
        f"{meta['intended_use']}\n\n"
        f"## Training Data\n"
        f"{meta['dataset']}\n\n"
        f"## Evaluation Metrics\n"
        f"- **ROC-AUC:** {float(meta['roc_auc']):.4f}\n"
        f"- **Brier Score:** {float(meta['brier_score']):.4f}\n\n"
        f"## Fairness & Limitations\n"
        f"{lim_text}\n"
    )
    ### END SOLUTION


In [ ]:
tutor.check("ch35_todo_03", generate_model_card_markdown)


---
## To-Do 35.4 (Cryptographic Bundle Builder): `build_reproducibility_bundle(artifact_path, config_dict, out_dir)`

Implement `build_reproducibility_bundle(artifact_path, config_dict, out_dir)`:
- Create `out_dir` (`mkdir(parents=True, exist_ok=True)`).
- Compute the lowercase 64-character SHA-256 hex digest of `artifact_path`.
- Write `config.json` and `manifest.json` inside `out_dir`, where `manifest` has exact keys: `artifact_name`, `artifact_bytes` (`int`), `artifact_sha256` (`str`), `python_version` (`str`), `created_utc` (`str`).
- Return the `manifest` dictionary.

In [ ]:
def build_reproducibility_bundle(artifact_path, config_dict, out_dir):
    """Hash artifact with SHA-256 and write config.json & manifest.json."""
    # STUB: raise NotImplementedError("Replace with your solution")
    ### BEGIN SOLUTION
    artifact = Path(artifact_path)
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    sha256 = hashlib.sha256(artifact.read_bytes()).hexdigest()
    manifest = {
        "artifact_name": artifact.name,
        "artifact_bytes": int(artifact.stat().st_size),
        "artifact_sha256": sha256,
        "python_version": platform.python_version(),
        "created_utc": datetime.now(timezone.utc).strftime(
            "%Y-%m-%dT%H:%M:%SZ"
        ),
    }
    (out / "config.json").write_text(
        json.dumps(config_dict, indent=2), encoding="utf-8"
    )
    (out / "manifest.json").write_text(
        json.dumps(manifest, indent=2), encoding="utf-8"
    )
    return manifest
    ### END SOLUTION


In [ ]:
tutor.check("ch35_todo_04", build_reproducibility_bundle)


---
## To-Do 35.5 (Integrity & Tamper Verifier): `verify_reproducibility_bundle(out_dir, artifact_path)`

Implement `verify_reproducibility_bundle(out_dir, artifact_path)`:
- Load `manifest.json` from `out_dir` (allowing `FileNotFoundError` if missing).
- Check whether `config.json` exists in `out_dir`.
- Recompute SHA-256 of `artifact_path` and compare against `manifest["artifact_sha256"]`.
- Return `{"hash_match": bool, "config_present": bool, "bundle_valid": bool}` (`hash_match and config_present`).

In [ ]:
def verify_reproducibility_bundle(out_dir, artifact_path):
    """Verify SHA-256 checksum and config.json presence in a bundle."""
    # STUB: raise NotImplementedError("Replace with your solution")
    ### BEGIN SOLUTION
    out = Path(out_dir)
    manifest = json.loads(
        (out / "manifest.json").read_text(encoding="utf-8")
    )
    actual_sha = hashlib.sha256(
        Path(artifact_path).read_bytes()
    ).hexdigest()
    hash_match = actual_sha == manifest["artifact_sha256"]
    config_present = (out / "config.json").exists()
    return {
        "hash_match": bool(hash_match),
        "config_present": bool(config_present),
        "bundle_valid": bool(hash_match and config_present),
    }
    ### END SOLUTION


In [ ]:
tutor.check("ch35_todo_05", verify_reproducibility_bundle)


---
## To-Do 35.6 (TRIPOD+AI Compliance Auditor): `score_tripod_ai_checklist(checklist_dict)`

Implement `score_tripod_ai_checklist(checklist_dict)` checking these 8 mandatory TRIPOD+AI boolean items:
`["calibration_reported", "code_released", "data_source_described", "leakage_prevented", "limitations_stated", "missing_data_handled", "seeds_fixed", "subgroup_fairness_audited"]`.
Return `{"score": int, "max_score": 8, "pct": round1, "missing_items": sorted_list}` without mutating `checklist_dict`.

In [ ]:
def score_tripod_ai_checklist(checklist_dict):
    """Score an 8-point TRIPOD+AI reporting compliance dictionary."""
    # STUB: raise NotImplementedError("Replace with your solution")
    ### BEGIN SOLUTION
    mandatory = [
        "calibration_reported",
        "code_released",
        "data_source_described",
        "leakage_prevented",
        "limitations_stated",
        "missing_data_handled",
        "seeds_fixed",
        "subgroup_fairness_audited",
    ]
    missing = sorted(
        item for item in mandatory
        if not bool(checklist_dict.get(item, False))
    )
    score = len(mandatory) - len(missing)
    return {
        "score": int(score),
        "max_score": 8,
        "pct": round(100.0 * score / 8, 1),
        "missing_items": missing,
    }
    ### END SOLUTION


In [ ]:
tutor.check("ch35_todo_06", score_tripod_ai_checklist)


---
## Annotated Debugging Post-Mortem (Solution Notebook Only)

- **To-Do 35.1 (Python's `bool` Is a Subclass of `int`):** Checking only `isinstance(val, (int, float))` lets `True` pass as `1` and `False` pass as `0`! Always guard `not isinstance(val, bool)` and `math.isfinite(float(val))` (rejecting `NaN`/`inf`), and build a fresh dict so caller state is never mutated.
- **To-Do 35.2 (Boundary Precision on Risk Tiers):** Use `>=` consistently (`prob >= self.threshold` for `"High"`, `prob >= self.threshold - 0.15` for `"Borderline"`, else `"Low"`) so exact boundary probabilities map deterministically across API and batch callers.
- **To-Do 35.4 & 35.5 (Cryptographic Integrity):** Reading raw bytes (`artifact.read_bytes()`) rather than text guarantees platform-independent SHA-256 checksums across Linux, macOS, and Windows. Letting `manifest.json` raise `FileNotFoundError` when absent prevents silent false-positive verification.

In [ ]:
tutor.summary()
